# Independent quadrupole input calibration

Fit `k_effective = scale * k_command + offset` independently for Q1, Q2, and Q3, together with Q1's transverse misalignment. Transforms live in the Torch wrapper's `transforms` ModuleDict, not on the underlying model. The wrapper applies them at the LUME `set()` boundary; Cheetah receives effective strengths as differentiable tensors.

The misalignment-only baseline uses the same data and training budget. This is a synthetic demonstration, not a guarantee of unique parameter identification. Varied scans and independent measurements are important when fitting correlated parameters.

This scan uses nonzero strengths: the currently tested Cheetah implementation gives nonfinite strength gradients at exactly `k1=0` in complex square-root backward. The wrapper does not clip inputs or suppress those errors.

In [ ]:
from copy import deepcopy
from itertools import product

import matplotlib.pyplot as plt
import torch

from example_model import create_example_model
from lume_applications.model_calibration import (
    TorchAffineTransform,
    TorchModelWithTransforms,
    calibrate_model,
)

torch.set_num_threads(1)
names = ("Q1_k", "Q2_k", "Q3_k")
nominal_model = create_example_model()
nominal_model.simulator.segment.SCR1.pixel_size = torch.tensor([2e-4, 2e-4])
reference_model = deepcopy(nominal_model)
true_scales = torch.tensor([1.08, 0.95, 1.06])
true_offsets = torch.tensor([0.03, -0.02, 0.04])
true_misalignment = torch.tensor([2e-4, -3e-4])
reference_model.simulator.segment.Q1.misalignment = true_misalignment.clone()

scan = torch.tensor(list(product([0.7, 1.0, 1.3], [-1.3, -1.0, -0.7], [0.7, 1.0, 1.3])))
training_inputs = {name: scan[:, i].clone() for i, name in enumerate(names)}
target_names = ["screen_image", "screen_centroid_x", "screen_centroid_y"]
records = {name: [] for name in target_names}
with torch.no_grad():
    for command in scan:
        effective = true_scales * command + true_offsets
        reference_model.set(dict(zip(names, effective)))
        for name, value in reference_model.get(target_names).items():
            records[name].append(value.detach().clone())
training_targets = {name: torch.stack(values) for name, values in records.items()}
image_scale = training_targets["screen_image"].abs().max()
if not torch.isfinite(image_scale) or image_scale <= 0:
    raise ValueError("Target image scale must be finite and positive")

## Fit identical models with and without transforms

Pass `TorchAffineTransform` instances when constructing `TorchModelWithTransforms`; initialization registers them and matches the model's dtype/device. This differentiable subclass of the general `AffineTransform` has learnable Torch parameters. The `transform_parameters` property supplies only trainable transform parameters for their optimizer group. Separate groups account for the different units of misalignment (meters) and strength calibration. `CalibrationModule`/`calibrate_model` need no modifications.

The learnable parameters are `scale` and `offset`. Scale can be positive or negative and may change sign between optimizer steps. Both affine transform classes reject zero scales at construction and evaluation; an optimizer step landing exactly on zero raises on the next evaluation rather than being clipped. For scale-only fitting, freeze `offset`; for offset-only fitting, freeze `scale`. Exclude frozen parameters from the optimizer.

In [ ]:
def calibration_loss(prediction, target):
    scale = image_scale if target.ndim == 2 else 1e-3
    return torch.nn.functional.mse_loss(prediction / scale, target / scale)


def fit_model(with_transforms):
    """Helper function to fit the model with or without parameter transforms."""

    # create a deep copy of the nominal model to avoid modifying the original
    model = deepcopy(nominal_model)

    # set up the misalignment parameter for the Q1 segment with lr = 1e-5
    quad = model.simulator.segment.Q1
    quad.misalignment = torch.nn.Parameter(torch.zeros_like(quad.misalignment))
    groups = [{"params": [quad.misalignment], "lr": 1e-5}]

    # apply parameter transforms if requested
    if with_transforms:
        # create a TorchModelWithTransforms wrapper around the model
        calibrated = TorchModelWithTransforms(
            model, transforms={name: TorchAffineTransform() for name in names}
        )
        # set learning rate for affine transform parameters to be 2e-3
        groups.append({"params": calibrated.transform_parameters, "lr": 2e-3})
    else:
        calibrated = model

    # print out the calibration parameters
    print("Calibration parameters:")
    for name, parameter in calibrated.named_parameters():
        print(f"{name}: {parameter.data}")
    print("\n")

    # set up the optimizer and start calibration
    optimizer = torch.optim.Adam(groups)
    history = calibrate_model(
        calibrated,
        training_inputs,
        training_targets,
        optimizer,
        steps=250,
        loss_fn=calibration_loss,
    )
    return calibrated, history


baseline_model, baseline_history = fit_model(False)
fitting_model, history = fit_model(True)
print(
    f"Misalignment-only loss: {baseline_history[0]:.6g} -> {baseline_history[-1]:.6g}"
)
print(f"Joint fit loss:         {history[0]:.6g} -> {history[-1]:.6g}")
assert history[-1] < history[0]
assert history[-1] < baseline_history[-1]

In [ ]:
for i, name in enumerate(names):
    fitted = fitting_model.transforms[name].get_transform_params()
    print(
        f"{name}: {fitted}; true scale={true_scales[i]:.4f}, offset={true_offsets[i]:.4f}"
    )
print("True misalignment [mm]:", (true_misalignment * 1e3).tolist())
print(
    "Fitted misalignment [mm]:",
    (fitting_model.model.simulator.segment.Q1.misalignment.detach() * 1e3).tolist(),
)

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
axes[0].semilogy(baseline_history, label="Misalignment only")
axes[0].semilogy(history, label="Joint fit")
axes[0].set(xlabel="Optimizer step", ylabel="Loss")
axes[0].legend()
for axis, attribute, truth in zip(
    axes[1:], ["scale", "offset"], [true_scales, true_offsets]
):
    learned = [
        getattr(fitting_model.transforms[name], attribute).item() for name in names
    ]
    axis.plot(names, truth.tolist(), "o", label="True")
    axis.plot(names, learned, "x", label="Fitted")
    axis.set_title(attribute)
    axis.legend()
fig.tight_layout()

## Use the calibrated model

Always send new commands through `fitting_model.set(...)` to apply `transform()`; calling `fitting_model.model.set(...)` bypasses transforms. `get()` applies inverse `untransform()` to configured variables, so input readbacks return command coordinates. Outputs without transforms remain unchanged.

Both simulator and transform parameters are included in `fitting_model.state_dict()`, with transforms under `transforms.` rather than `model.transforms.`. To reload, construct an equivalent model and pass the same transform structure to the wrapper before calling `load_state_dict()`. For a non-Torch LUME model, use `ModelWithTransforms(model, {name: transform_instance})` instead; each instance must implement `transform(value)` and inverse `untransform(value)`. 